In [19]:
%pip install requests
import sys
from pathlib import Path

# Add project root (Pokebot/) to Python path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print("Project root added to path:", PROJECT_ROOT)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Project root added to path: c:\Users\ericn\PokeBot



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import pandas as pd

# Load data from previous notebooks
products = pd.read_csv("../data/products.csv")
drop_events = pd.read_csv("../data/drop_events.csv")
drop_products = pd.read_csv("../data/drop_products.csv")

products.head(), drop_events.head(), drop_products.head()

(   product_id                product_name    sku_upc product_type retailer  \
 0           2  Paldea Evolved Booster Box  987654321  Booster Box   Target   
 
      msrp  exclusive_flag       release_wave                 notes  
 0  161.64           False  SV Paldea Evolved  Standard booster box  ,
    drop_id retailer source  zip_code  price_observed  \
 0        1   Target    app     60491           49.99   
 
                   observed_at                       notes  
 0  2026-03-27T20:16:04.167692  ETB restock via Target app  ,
    drop_id  product_id
 0        1           1)

In [21]:
LOCAL_ZIPS = {"60491", "60441", "60451", "60462"}

In [22]:
drop_detail = (
    drop_events[[
        "drop_id",
        "retailer",
        "source",
        "zip_code",
        "price_observed",
        "observed_at"
    ]]
    .merge(drop_products, on="drop_id", how="left")
    .merge(
        products[[
            "product_id",
            "product_name",
            "product_type",
            "exclusive_flag",
            "msrp"
        ]],
        on="product_id",
        how="left"
    )
)

In [23]:
# Rename retailer columns for clarity
drop_detail = drop_detail.rename(columns={
    "retailer_x": "drop_retailer",
    "retailer_y": "product_retailer"
})

drop_detail.columns

Index(['drop_id', 'retailer', 'source', 'zip_code', 'price_observed',
       'observed_at', 'product_id', 'product_name', 'product_type',
       'exclusive_flag', 'msrp'],
      dtype='object')

In [24]:
# Load store patterns
store_patterns = pd.read_csv("../data/store_patterns.csv")

# Build lookup: (retailer, zip_code) -> reliability
store_reliability = {
    (r, z): s
    for r, z, s in zip(
        store_patterns["retailer"],
        store_patterns["zip_code"].astype(str),
        store_patterns["store_reliability_score"]
    )
}

def score_drop(row):
    score = 0
    if row["exclusive_flag"]:
        score += 3
    if row["product_type"] in ["ETB", "Booster Box"]:
        score += 2
    if row["retailer"] == "Pokemon Center":
        score += 2
    if row["source"] in ["email", "app", "in_person"]:
        score += 1
    if row["zip_code"] in LOCAL_ZIPS and row["price_observed"] <= row["msrp"]:
        score += 5

    # ✅ Store reliability bias (lightweight, capped)
    score += min(store_reliability.get((row["retailer"], str(row["zip_code"])), 0), 3)
    return score

In [25]:
drop_detail["drop_score"] = drop_detail.apply(score_drop, axis=1)

scored_drops = drop_detail.sort_values(
    "drop_score", ascending=False
)

scored_drops

,drop_id,retailer,source,zip_code,price_observed,observed_at,product_id,product_name,product_type,exclusive_flag,msrp,drop_score
0,1,Target,app,60491,49.99,2026-03-27T20:16:04.167692,1,NaN,NaN,NaN,NaN,5.5


In [26]:
scored_drops.to_csv("../data/scored_drops.csv", index=False)

print("✅ scored_drops.csv saved")

✅ scored_drops.csv saved


In [27]:
from src.notify_discord import send_discord

WEBHOOK_URL = "https://discord.com/api/webhooks/1486944657719169215/Cicg4EW7BKInSO0ZhUwHVeoEyCZVTbTF7bKqJgHMcvy3LTe-Ye5fe6heyb5mq7caj96z"
ALERT_THRESHOLD = 0  # tune later

alerts = scored_drops[scored_drops["drop_score"] >= ALERT_THRESHOLD]

for _, row in alerts.iterrows():
    title = f"🔥 High‑Priority Drop: {row['product_name']}"
    message = (
        f"Retailer: {row['retailer']}\n"
        f"Source: {row['source']}\n"
        f"Score: {row['drop_score']}\n"
        f"Notes: {row.get('notes', '')}"
    )

    send_discord(WEBHOOK_URL, title, message)

print(f"✅ {len(alerts)} Discord alert(s) sent")

✅ 1 Discord alert(s) sent
